In [2]:
"""
MVN 常態性檢定 — Cramér-Wold 精神的實作版
"""
import numpy as np
from scipy import stats
import pingouin as pg

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

# ============================================================
# Case 1: 真的 MVN 資料
# ============================================================
mu = [0, 0, 0]
Sigma = np.array([[1.0, 0.5, 0.3],
                  [0.5, 1.0, 0.2],
                  [0.3, 0.2, 1.0]])
X_mvn = rng.multivariate_normal(mu, Sigma, size=500)

# ============================================================
# Case 2: 邊際 normal 但聯合不是 MVN
# 經典反例: Y = X²·sign(U), 仍是 N(0,1) 邊際但跟 X 非線性相關
# ============================================================
n = 500
X1 = rng.normal(0, 1, n)
X2 = rng.normal(0, 1, n)
# X3 強制由 X1, X2 的非線性關係決定 → 破壞聯合 normality
X3 = np.sign(X1 * X2) * np.abs(rng.normal(0, 1, n))
X_fake = np.column_stack([X1, X2, X3])

# ============================================================
# Method A: Henze-Zirkler test (pingouin 內建)
# ============================================================
print("=" * 60)
print("Method A: Henze-Zirkler MVN test (industry standard)")
print("=" * 60)

hz_stat, hz_p, hz_normal = pg.multivariate_normality(X_mvn, alpha=0.05)
print(f"\n真 MVN 資料:")
print(f"  H-Z statistic = {hz_stat:.4f}, p-value = {hz_p:.4f}")
print(f"  Is MVN? {hz_normal}  ← 應為 True")

hz_stat, hz_p, hz_normal = pg.multivariate_normality(X_fake, alpha=0.05)
print(f"\n假 MVN (邊際 normal,聯合不是):")
print(f"  H-Z statistic = {hz_stat:.4f}, p-value = {hz_p:.4f}")
print(f"  Is MVN? {hz_normal}  ← 應為 False")


Method A: Henze-Zirkler MVN test (industry standard)

真 MVN 資料:
  H-Z statistic = 0.9100, p-value = 0.2613
  Is MVN? True  ← 應為 True

假 MVN (邊際 normal,聯合不是):
  H-Z statistic = 4.6574, p-value = 0.0000
  Is MVN? False  ← 應為 False


In [ ]:
# ============================================================
# Method B: 邊際 Shapiro-Wilk (necessary but NOT sufficient!)
# ============================================================
print("\n" + "=" * 60)
print("Method B: 邊際 Shapiro-Wilk (容易被騙)")
print("=" * 60)

print(f"\n真 MVN 資料的邊際:")
for i in range(3):
    stat, p = stats.shapiro(X_mvn[:, i])
    print(f"  X{i+1}: p = {p:.4f}  {'(normal)' if p > 0.05 else '(NOT normal)'}")

print(f"\n假 MVN 資料的邊際:")
for i in range(3):
    stat, p = stats.shapiro(X_fake[:, i])
    print(f"  X{i+1}: p = {p:.4f}  {'(normal)' if p > 0.05 else '(NOT normal)'}")
print("→ 假資料每個邊際都『看起來』像 normal, 但 H-Z 已經抓到聯合分佈不對")

# ============================================================
# Method C: 手工版「Cramér-Wold 精神」測試
# 隨機投影到 100 個方向, 每個方向跑 Shapiro-Wilk
# ============================================================
print("\n" + "=" * 60)
print("Method C: 手工 Cramér-Wold 風格 (投影到很多方向)")
print("=" * 60)

def cramer_wold_style_test(X, n_directions=100, alpha=0.05):
    """投影到很多隨機方向,看『有多少方向看起來不 normal』"""
    p = X.shape[1]
    n_fail = 0
    p_values = []
    for _ in range(n_directions):
        # 從單位球面均勻抽一個方向
        a = rng.normal(0, 1, p)
        a /= np.linalg.norm(a)
        # 投影
        projection = X @ a
        # 對 1D 投影跑 Shapiro-Wilk
        _, p_val = stats.shapiro(projection)
        p_values.append(p_val)
        if p_val < alpha:
            n_fail += 1
    return n_fail / n_directions, np.median(p_values)

frac_fail, med_p = cramer_wold_style_test(X_mvn)
print(f"\n真 MVN: {frac_fail*100:.1f}% 方向被拒絕 (期望 ~5%), median p = {med_p:.3f}")

frac_fail, med_p = cramer_wold_style_test(X_fake)
print(f"假 MVN: {frac_fail*100:.1f}% 方向被拒絕, median p = {med_p:.3f}")
print("→ Cramér-Wold 精神:很多方向不 normal → 整體不 MVN")


Method B: 邊際 Shapiro-Wilk (容易被騙)

真 MVN 資料的邊際:
  X1: p = 0.9070  (normal)
  X2: p = 0.3225  (normal)
  X3: p = 0.5930  (normal)

假 MVN 資料的邊際:
  X1: p = 0.9210  (normal)
  X2: p = 0.3365  (normal)
  X3: p = 0.4997  (normal)
→ 假資料每個邊際都『看起來』像 normal, 但 H-Z 已經抓到聯合分佈不對

Method C: 手工 Cramér-Wold 風格 (投影到很多方向)

真 MVN: 12.0% 方向被拒絕 (期望 ~5%), median p = 0.541
假 MVN: 65.0% 方向被拒絕, median p = 0.009
→ Cramér-Wold 精神:很多方向不 normal → 整體不 MVN


In [2]:
"""
Mahalanobis vs χ² Q-Q plot — MVN 假設的視覺化診斷
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
n = 500
p = 3

# ============================================================
# Case 1: 真 MVN 資料
# ============================================================
mu = [0, 0, 0]
Sigma = np.array([[1.0, 0.5, 0.3],
                  [0.5, 1.0, 0.2],
                  [0.3, 0.2, 1.0]])
X_mvn = rng.multivariate_normal(mu, Sigma, size=n)

# ============================================================
# Case 2: Multivariate t-distribution (heavy tails)
# 這是 MVN 失效的最常見情境 — 形狀類似 MVN 但 tails 比較重
# ============================================================
df_t = 3  # 自由度越小, tail 越重
# Multivariate t: X = mu + Z / sqrt(W/df), Z ~ MVN(0, Sigma), W ~ chi^2(df)
Z = rng.multivariate_normal([0, 0, 0], Sigma, size=n)
W = rng.chisquare(df_t, size=n)
X_fake = Z / np.sqrt(W[:, None] / df_t)

# ============================================================
# 計算 Mahalanobis d² 和理論 χ² 分位數
# ============================================================
def mahalanobis_qq(X):
    """
    回傳 Q-Q plot 所需的兩條軸:
    - 理論 χ²(p) 分位數 (x 軸)
    - 觀察到的 Mahalanobis d² 排序後 (y 軸)
    """
    n, p = X.shape
    Xc = X - X.mean(axis=0)
    S = np.cov(X, rowvar=False)
    S_inv = np.linalg.inv(S)
    
    # d²ᵢ = (xᵢ - x̄)ᵀ S⁻¹ (xᵢ - x̄)
    d_sq = np.einsum('ni,ij,nj->n', Xc, S_inv, Xc)
    
    # 排序觀察值
    d_sq_sorted = np.sort(d_sq)
    
    # 理論分位數: 第 i 個位置對應 (i - 0.5) / n 分位點
    quantile_positions = (np.arange(1, n+1) - 0.5) / n
    theoretical = stats.chi2.ppf(quantile_positions, df=p)
    
    return theoretical, d_sq_sorted

# ============================================================
# 畫圖
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, X, title, expected in [
    (axes[0], X_mvn, "Case A: True MVN data", "points should hug the line"),
    (axes[1], X_fake, "Case B: Multivariate t (df=3)\nheavy-tailed, NOT MVN", "upper-right curves upward")
]:
    theoretical, observed = mahalanobis_qq(X)
    
    # 散點
    ax.scatter(theoretical, observed, alpha=0.4, s=20, color='steelblue', edgecolor='none')
    
    # y = x 對角參考線
    lims = [0, max(theoretical.max(), observed.max()) * 1.05]
    ax.plot(lims, lims, 'r--', lw=2, label='y = x (perfect MVN)')
    
    # 95% χ² 門檻 (outlier 判定線)
    threshold = stats.chi2.ppf(0.95, df=p)
    ax.axvline(threshold, color='gray', linestyle=':', alpha=0.7,
               label=f'95% χ²({p}) threshold = {threshold:.2f}')
    ax.axhline(threshold, color='gray', linestyle=':', alpha=0.7)
    
    ax.set_xlabel(f'Theoretical χ²({p}) quantiles')
    ax.set_ylabel('Observed Mahalanobis d² (sorted)')
    ax.set_title(f'{title}\n→ {expected}')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect('equal')

plt.suptitle('Mahalanobis vs Chi-squared Q-Q plot for MVN diagnosis', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/Users/fangsiyu/Desktop/sdu-2026-code/805_multivariate_statistical_analysis/note/fig5_mvn_qq.png', dpi=110, bbox_inches='tight')
plt.close()
print("→ 圖存到 fig_mvn_qq.png")

# ============================================================
# 數值報告
# ============================================================
print("\n" + "=" * 60)
print("怎麼讀這個圖")
print("=" * 60)
print("""
1. X 軸: 理論上,如果資料是 MVN, d² 應該服從 χ²(p)。
   把 χ²(p) 的分位數當『標準答案』畫在 x 軸上。

2. Y 軸: 把實際算出的 d² 從小到大排序,當作觀察值。

3. 解讀:
   - 點貼著紅色 y=x 線 → 資料是 MVN ✓
   - 點在右上方偏離向上彎 → heavy tails (比 MVN 多 outliers)
   - 點在右上方偏離向下彎 → light tails (比 MVN 少極端值)
   - 中段彎曲 → 整體分布形狀不對
""")

# 數值診斷
for X, name in [(X_mvn, "Case A (真 MVN)"), (X_fake, "Case B (multivariate t, df=3)")]:
    Xc = X - X.mean(axis=0)
    S_inv = np.linalg.inv(np.cov(X, rowvar=False))
    d_sq = np.einsum('ni,ij,nj->n', Xc, S_inv, Xc)
    print(f"{name}:")
    print(f"  Mean(d²)   = {d_sq.mean():.3f}   (MVN 理論 = p = {p})")
    print(f"  Var(d²)    = {d_sq.var():.3f}   (MVN 理論 = 2p = {2*p})")
    print(f"  Median(d²) = {np.median(d_sq):.3f}  (MVN 理論 ≈ {stats.chi2.ppf(0.5, df=p):.3f})")
    print()

→ 圖存到 fig_mvn_qq.png

怎麼讀這個圖

1. X 軸: 理論上,如果資料是 MVN, d² 應該服從 χ²(p)。
   把 χ²(p) 的分位數當『標準答案』畫在 x 軸上。

2. Y 軸: 把實際算出的 d² 從小到大排序,當作觀察值。

3. 解讀:
   - 點貼著紅色 y=x 線 → 資料是 MVN ✓
   - 點在右上方偏離向上彎 → heavy tails (比 MVN 多 outliers)
   - 點在右上方偏離向下彎 → light tails (比 MVN 少極端值)
   - 中段彎曲 → 整體分布形狀不對

Case A (真 MVN):
  Mean(d²)   = 2.994   (MVN 理論 = p = 3)
  Var(d²)    = 6.407   (MVN 理論 = 2p = 6)
  Median(d²) = 2.275  (MVN 理論 ≈ 2.366)

Case B (multivariate t, df=3):
  Mean(d²)   = 2.994   (MVN 理論 = p = 3)
  Var(d²)    = 34.353   (MVN 理論 = 2p = 6)
  Median(d²) = 1.315  (MVN 理論 ≈ 2.366)

